## Extracción de datos de empresas disueltas desde la API del INE

### Objetivo
Este notebook extrae y estructura los datos de **sociedades mercantiles disueltas** en España a través de la API pública TEMPUS del INE (tabla 13915).

Los datos se desglosan por **territorio**, **causa de disolución** (Voluntaria, Por fusión, Otras) y **periodo mensual**, obteniendo el número de sociedades disueltas en cada segmento.

### Metodología
1. **Conexión a la API del INE** — Llamada al endpoint `/DATOS_TABLA/13915`.
2. **Extracción y filtrado** — Recorrido de la respuesta JSON excluyendo registros nacionales y agrupaciones "Total" para evitar duplicados.
3. **Estructuración** — Poblado de un diccionario con las columnas: `id_dis`, `territorio`, `id_tiempo`, `razon` y `numero_sociedades`.
4. **Exportación** — Volcado a CSV en `../files/data_raw/empresas_disueltas.csv`.

### Contexto del proyecto
Estos datos se integran en un análisis de **resiliencia empresarial en España**, donde se cruzarán con constituciones de empresas e IPC para estudiar la relación entre el entorno macroeconómico y la mortalidad empresarial.

In [ ]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación del módulo de conexión a la API
from src.api import conection_api
from src.api.config import API_URLS

In [ ]:
url_dis = API_URLS["disueltas"]    #Disueltas

In [ ]:
data_dis = conection_api.llamada_api(url_dis)

In [ ]:
len(data_dis)

In [ ]:
data_dis[70]

In [ ]:
for dato in data_dis:
    for serie in dato['Data']:
        print(f'{dato['Nombre']}, {dato['FK_Escala']}, {serie['FK_Periodo']}, {serie['Anyo']}, {serie['Valor']}')

In [ ]:
for serie in data_dis[:5]:
    print(serie['Nombre'])

In [ ]:
empresas_disueltas = { 
    'id_dis': [], 
    'territorio': [], 
    'id_tiempo': [],
    'razon': [], 
    'numero_sociedades': []            
}
contador = 1

for serie in data_dis:
    nombre_completo = serie['Nombre']
    
    if "nacional" not in nombre_completo.lower():
        
        partes = nombre_completo.split('.')
        territorio = partes[0].strip()
        razon = partes[1].strip()

        # CONDIClÓN AÑADIDA: Solo procesa y appendea si la razón NO es "Total"
        if razon.lower() != "total":

            for data in serie['Data']:
                id_tiempo = str(data['Anyo']) + str(data['FK_Periodo']).zfill(2)

                empresas_disueltas['id_dis'].append(contador)
                empresas_disueltas['territorio'].append(territorio)
                empresas_disueltas['id_tiempo'].append(id_tiempo)
                empresas_disueltas['razon'].append(razon)
                empresas_disueltas['numero_sociedades'].append(int(data['Valor']))
                            
                contador += 1

In [ ]:
empresas_disueltas

In [ ]:
pd.DataFrame(empresas_disueltas).to_csv('../files/data_raw/empresas_disueltas.csv', index=False)